# E1 - segmentacja: przegląd wyników

Czyta gotowe pliki pamięci podręcznej i pokazuje, jak wyglądają kolekcje fragmentów po trzech strategiach segmentacji oraz jaki materiał wchodzi do korpusu. Nie liczy nic na nagraniach. Odcinki bez zakresów albo bez bufora pomija i wypisuje.

**Wymaga:** zbudowanej segmentacji:

```powershell
python scripts/run_segmentation.py configs/e1a_tbbt.yaml configs/e1a_office.yaml
python scripts/run_segmentation.py configs/e1b_tbbt.yaml configs/e1b_office.yaml
python scripts/run_segmentation.py configs/e1c_tbbt.yaml configs/e1c_office.yaml
```

Stałe okna liczą się natychmiast; oba detektory ujęć dekodują każdą klatkę nagrania (kilka minut na odcinek, wznawialne per odcinek).

**Zapisuje:** nic, tylko wypisuje.

In [ ]:
import csv
import json
import sys
from collections import defaultdict
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.segmentation import segments as seg

SERIES = ("tbbt", "office")
STRATEGIES = ("fixed_window", "shots_histogram", "shots_transnetv2")

# The strategy E1 settled on, and the one the appendix of the thesis reports as
# the collection every later experiment is measured over. The survey of the
# material below describes THAT collection; the comparison of all three, which
# is what E1 itself is, stays on STRATEGIES.
FROZEN_STRATEGY = "shots_histogram"      # E1-B

# Which part the tables below describe. The statistics of section 2 were written
# for the test part and read as if that were the only choice; it is a parameter.
SPLIT = "test"


def read_ranges(series):
    path = ROOT / "data" / "interim" / series / f"{series}_ranges.csv"
    if not path.exists():
        return {}
    episodes = {}
    with open(path, encoding="utf-8-sig", newline="") as f:
        for r in csv.DictReader(f, delimiter=";"):
            e = episodes.setdefault(r["episode"], {"split": r["split"], "corpus": 0.0})
            e["corpus"] += float(r["duration"])
    return episodes


def read_segments(series):
    out = {}
    for strategy in STRATEGIES:
        path = ROOT / "data" / "cache" / "segmentation" / strategy / f"{series}_segments.csv"
        if path.exists():
            out[strategy] = seg.load(path)
    return out


def read_queries(series):
    out = []
    for split in ("dev", "test"):
        path = ROOT / "data" / "annotations" / series / f"{series}_queries_{split}.jsonl"
        if not path.exists():
            continue
        with open(path, encoding="utf-8") as f:
            for line in f:
                q = json.loads(line)
                q["split"] = split
                q["episode"] = q["vid_name"].split("_", 1)[1]
                out.append(q)
    return out


def mmss(x):
    s = round(x)
    return f"{s // 60}:{s % 60:02d}"


def hhmmss(x):
    s = round(x)
    return f"{s // 3600}:{s % 3600 // 60:02d}:{s % 60:02d}"


RANGES = {s: read_ranges(s) for s in SERIES}
SEGMENTS = {s: read_segments(s) for s in SERIES}
QUERIES = {s: read_queries(s) for s in SERIES}

for s in SERIES:
    have = {k: len({r["episode"] for r in v}) for k, v in SEGMENTS[s].items()}
    print(f"{s:<8} ranges: {len(RANGES[s]):>3} episodes | segments: {have or '-'}"
          f" | queries: {len(QUERIES[s])}")

## 1. Wykaz materiału badawczego

Sezon, odcinek, czas w korpusie (`mm:ss`, po wycięciu czołówek i napisów), liczba fragmentów przy stałych oknach oraz liczba zapytań. Sumy per zbiór w `hh:mm:ss`. Tytuły odcinków są w `work/*_episode_titles.csv`.

In [ ]:
for series in SERIES:
    ranges, queries = RANGES[series], QUERIES[series]
    segments = SEGMENTS[series].get(FROZEN_STRATEGY, [])
    if not ranges or not segments:
        print(f"{series}: no ranges or no {FROZEN_STRATEGY} cache - skipped\n")
        continue
    fragments = defaultdict(int)
    for r in segments:
        fragments[r["episode"]] += 1
    query_count = defaultdict(int)
    for q in queries:
        query_count[q["episode"]] += 1

    print(f"=== {series} ===")
    header = f"{'episode':<9}{'split':<7}{'corpus':>9}{'fragments':>11}{'queries':>9}"
    for split in ("dev", "test"):
        eps = sorted(ep for ep, e in ranges.items() if e["split"] == split)
        if not eps:
            continue
        print(f"-- {split} " + "-" * (len(header) - len(split) - 4))
        for ep in eps:
            missing = "" if ep in fragments else "  <- no segments"
            print(f"{ep:<9}{split:<7}{mmss(ranges[ep]['corpus']):>9}"
                  f"{fragments.get(ep, 0):>11}{query_count.get(ep, 0):>9}{missing}")
        total = sum(ranges[ep]["corpus"] for ep in eps)
        print(f"{'total':<9}{split:<7}{hhmmss(total):>9}"
              f"{sum(fragments.get(ep, 0) for ep in eps):>11}"
              f"{sum(query_count.get(ep, 0) for ep in eps):>9}")
    print()

## 2. Statystyka segmentacji

Część testowa materiału, per serial i strategia: liczba fragmentów, średnia długość fragmentu oraz udział zdarzeń zajmujących więcej niż jeden fragment (zdarzenie to przedział `ts` zapytania testowego, fragment zajęty to niezerowe przecięcie ze zdarzeniem).

In [ ]:
header = (f"{'series':<9}{'strategy':<19}{'fragments':>10}{'mean [s]':>10}"
          f"{'events':>8}{'>1 fragment':>13}")
print(header)
print("-" * len(header))
for series in SERIES:
    events = [q for q in QUERIES[series] if q["split"] == SPLIT]
    for strategy in STRATEGIES:
        rows = [r for r in SEGMENTS[series].get(strategy, []) if r["split"] == SPLIT]
        if not rows:
            print(f"{series:<9}{strategy:<19}{'-':>10}{'-':>10}{'-':>8}{'-':>13}")
            continue
        by_episode = defaultdict(list)
        for r in rows:
            by_episode[r["episode"]].append(r)
        spanning = total_events = 0
        for q in events:
            overlapping = sum(1 for r in by_episode.get(q["episode"], [])
                              if r["start"] < q["ts"][1] and r["end"] > q["ts"][0])
            if overlapping:
                total_events += 1
                spanning += overlapping > 1
        mean = sum(r["duration"] for r in rows) / len(rows)
        share = f"{100 * spanning / total_events:.1f}%" if total_events else "-"
        print(f"{series:<9}{strategy:<19}{len(rows):>10}{mean:>10.2f}"
              f"{total_events:>8}{share:>13}")

## 3. Kolekcje zbioru deweloperskiego

Liczebność przeszukiwanych kolekcji na części deweloperskiej, per strategia.

In [ ]:
header = f"{'series':<9}" + "".join(f"{s:>19}" for s in STRATEGIES)
print(header)
print("-" * len(header))
for series in SERIES:
    counts = []
    for strategy in STRATEGIES:
        rows = [r for r in SEGMENTS[series].get(strategy, []) if r["split"] == "dev"]
        counts.append(f"{len(rows) or '-':>19}")
    print(f"{series:<9}" + "".join(counts))